In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import json

In [10]:
base_model = "models/gemma-2b-it"  # same as training

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.78it/s]


In [11]:
model = PeftModel.from_pretrained(model, "./models/gemma-2b-it-fine-tuned")

In [12]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k

In [13]:
SYSTEM_INSTRUCTION = """
You are an AI system designed to transform raw, unstructured job descriptions into structured, professional, and ATS-friendly job descriptions.

Your task:
- Extract AND rewrite content into a polished, professional format.
- Expand short or vague statements into clear, detailed, and actionable bullet points.
- Improve grammar, clarity, and tone while preserving original meaning.

Rules:
- Output MUST be valid JSON only.
- Follow the exact schema provided.
- Do NOT include explanations or extra text.
- Do NOT hallucinate unrealistic details.
- Do NOT copy sentences directly from input — always rewrite them professionally.

Enhancement Rules:
- Convert short phrases into complete, professional sentences.
- Add clarity by specifying intent (e.g., “handle tickets” → “resolve IT support tickets efficiently within defined SLAs”).
- Use strong action verbs (e.g., manage, ensure, deliver, coordinate, analyze).
- Maintain ATS-friendly language with relevant keywords.
- Avoid vague wording like “do”, “work on”, “handle”.
- Only include information explicitly present in the input.
- Do NOT infer industry or qualifications unless clearly mentioned.

Writing Style:
- Use concise but complete sentences.
- Each bullet point should be meaningful and self-contained.
- Maintain consistency across all sections.
- Generate ONLY ONE JSON object.
- Stop immediately after closing }.
"""

OUTPUT_SCHEMA = """
[
  "job_title": "",
  "location": "",
  "industry": "",
  "responsibilities": [],
  "requirements": [],
  "qualifications": [],
  "experience": [],
  "other_requirements": []
]
"""

def prompt(content):
  return f"""
### SYSTEM:
{SYSTEM_INSTRUCTION}

### USER:
Convert the following raw job description into structured JSON.

### Expected OUTPUT FORMAT:
Return a fully populated JSON following this schema:
{OUTPUT_SCHEMA}

### INPUT:
{content}

"""

In [14]:
content = "we need python dev.. 2+ yrs exp!!! backend work"
inputs = tokenizer(prompt(content), return_tensors="pt").to(model.device)

In [15]:
input_length = inputs["input_ids"].shape[1]

max_new_tokens = min(768, max(256, int(input_length * 1.2)))

outputs = model.generate(
    **inputs,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    repetition_penalty = 1.2
)

In [16]:
def extract_first_json(output: str):
    start = output.find("{")
    if start == -1:
        return "{}"

    brace_count = 0
    for i in range(start, len(output)):
        if output[i] == "{":
            brace_count += 1
        elif output[i] == "}":
            brace_count -= 1

        if brace_count == 0:
            return output[start:i+1]

    return "{}"


def enforce_input_constraints(data, content):
    if isinstance(data, str):
        data = json.loads(data)  # convert string → dict

    raw_input_lower = content.lower()

    if "industry" not in raw_input_lower:
        data["industry"] = ""

    if not any(word in raw_input_lower for word in ["degree", "bachelor", "master", "phd"]):
        data["qualifications"] = ""

    return data

In [17]:
output = tokenizer.decode(outputs[0], skip_special_tokens=True)

json_text = extract_first_json(output)
#data = enforce_input_constraints(json_text, content)

print(json_text)

{"job_title": "Python Developer", "location": "N/A", "industry": "Multinational Technology Provider", "responsibilities": ["Design, develop, and maintain robust Python applications for backend services.", "Collaborate with cross-functional teams including product managers, architects, and QA engineers to define project requirements and technical specifications.", "Write clean, efficient, and well-documented code adhering to best practices and coding standards.", "Integrate third-party APIs and libraries as needed to enhance application functionality.", "Troubleshoot and resolve complex issues related to performance bottlenecks, security vulnerabilities, and data integrity.", "Participate in code reviews, provide constructive feedback, and contribute to continuous improvement initiatives.", "Stay updated with emerging technologies and trends in software development to implement innovative solutions."], "requirements": ["Minimum of 2 years of experience in Python programming.", "Proficie

{'job_title': 'Python Developer', 
'location': 'N/A', 
'industry': '', 
'responsibilities': ['Design, develop, and maintain robust Python applications for our clients.', 'Collaborate closely with cross-functional teams to define project requirements and timelines.', 'Write clean, efficient, and well-documented code that adheres to best practices.', 'Troubleshoot and resolve technical issues related to application performance and functionality.', 'Participate in code reviews and provide constructive feedback to peers.', 'Ensure high quality deliverables through rigorous testing and validation procedures.'], 
'requirements': ['Minimum of 2 years of experience as a Python developer.', 'Proficiency in Object Oriented Programming principles and data structures.', 'Strong understanding of web services and RESTful APIs.', 'Experience with popular frameworks such as Django or Flask is preferred.', 'Excellent communication skills and ability to collaborate effectively with team members.'], 
'qualifications': '', 
'experience': ['At least 2 years of experience developing software solutions using Python.', 'Familiarity with cloud computing platforms like AWS or Azure is desirable.'], 
'other_requirements': ['Ability to adapt to changing priorities and deadlines.', 'Commitment to continuous learning and staying updated with emerging technologies.']}

{"job_title": "Python Developer", 
"location": "N/A", 
"industry": "Multinational Technology Provider", 
"responsibilities": ["Design, develop, and maintain robust Python applications for our clients.", "Collaborate closely with cross-functional teams to define project requirements and timelines.", "Write clean, efficient, and well-documented code that adheres to best practices.", "Troubleshoot and resolve technical issues related to application performance and functionality.", "Participate in code reviews and provide constructive feedback to peers.", "Ensure high quality deliverables through rigorous testing and validation procedures."], 
"requirements": ["Minimum of 2 years of experience as a Python developer.", "Proficiency in Object Oriented Programming principles and data structures.", "Strong understanding of web services and RESTful APIs.", "Experience with popular frameworks such as Django or Flask is preferred.", "Excellent communication skills and ability to collaborate effectively with team members."], 
"qualifications": ["Bachelor’s degree in Computer Science or equivalent.", "Relevant certifications may be advantageous."], 
"experience": ["At least 2 years of experience developing software solutions using Python.", "Familiarity with cloud computing platforms like AWS or Azure is desirable."], 
"other_requirements": ["Ability to adapt to changing priorities and deadlines.", "Commitment to continuous learning and staying updated with emerging technologies."]}